# S3 : reference E0, XGBoost transactionnel

Ce notebook entraine E0 dans les scenarios A et B, sur la population principale (TRANSFER + CASH_OUT) et la population secondaire (tous les types), puis genere la note pour le prof dans `docs/note_s3_E0.md`.

On suit la fiche protocole : decoupage par heures, reglages sur la validation seulement, seuil gele sur la validation, test utilise une seule fois.

Lancer d'abord le notebook `02_controle_s2.ipynb` dans la meme session si on veut que la note reprenne le tableau de concordance. Prevoir un runtime avec assez de RAM (le CSV fait 6,36 millions de lignes).

In [ ]:
!git clone -q https://github.com/kenzaez/pfa06-graph-fraud.git
%cd pfa06-graph-fraud
!pip install -q -r requirements.txt

## 1. Donnees

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!kaggle datasets download -d ealaxi/paysim1 -p data
!unzip -o -q data/paysim1.zip -d data

In [ ]:
from src.audit import load, sha256

PATH = "data/PS_20174392719_1491204439457_log.csv"
assert sha256(PATH) == "16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b"
df = load(PATH)
df.shape

## 2. Tests du protocole avant d'entrainer

Si un test casse, on s'arrete la.

In [ ]:
!python -m pytest -q tests/test_protocol.py tests/test_features.py

## 3. Entrainement et evaluation

Pour chaque scenario, on compare sur la validation quatre facons de gerer le desequilibre (ratio naturel avec `scale_pos_weight`, ou sous echantillonnage a 1 %, 5 %, 10 % de fraudes) et la grille profondeur x taux d'apprentissage, avec arret precoce. On garde la meilleure AP de validation. La population secondaire reprend les memes hyperparametres.

Ca prend un moment, surtout les fits au ratio naturel sur 2,6 millions de lignes.

In [ ]:
import warnings
import pandas as pd
from src import e0

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
tab, par_jour, choix = e0.main(df)
choix

## 4. Resultats

Le tableau complet est dans `reports/e0/resultats_e0.csv`. Ici on regarde la population principale.

In [ ]:
cols = ["modele", "scenario", "partition", "ap", "taux_pct", "ratio_ap_taux", "P@100", "R@100", "P@250", "R@250", "P@500", "R@500", "precision", "rappel", "f1", "brier"]
tab[tab.population == "population_principale"][cols].round(4)

In [ ]:
from IPython.display import Image, display

for s in ["A", "B"]:
    display(Image(f"reports/e0/pr_{s}_population_principale.png"))
    display(Image(f"reports/e0/fiabilite_{s}_population_principale.png"))

In [ ]:
par_jour[par_jour.population == "population_principale"].round(4)

## 5. Note pour le prof

La note est generee a partir des fichiers de resultats, donc les chiffres sont forcement ceux du code.

In [ ]:
from src import rapport_s3
from IPython.display import Markdown

Markdown(rapport_s3.main().read_text())

## 6. Tout recuperer

On telecharge le zip, puis on met ces fichiers dans le depot et on pousse sur GitHub.

In [ ]:
!zip -q -r livrable_s3.zip docs reports models configs
from google.colab import files

files.download("livrable_s3.zip")